# Notebook 5: Governance and promotion gates

This notebook demonstrates how the repo separates candidate memory from trusted memory by applying explicit promotion rules before something is considered eligible.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from src.claims.extractor import ClaimExtractor
from src.governance.promotion import PromotionEngine
from src.governance.rules import (
    MinimumClaimsRule,
    MinimumConfidenceRule,
    MinimumMaturityRule,
    NoContradictionsRule,
)
from src.normalization.mapper import ClaimToMemoryMapper

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-05"
shutil.rmtree(memory_root, ignore_errors=True)
extractor = ClaimExtractor()
mapper = ClaimToMemoryMapper()
engine = PromotionEngine()

In [ ]:
claims = extractor.extract(
    "Acme Payments restored the previous certificate and checkout recovered for enterprise customers.",
    source_ref="promotion-demo",
)
candidate = mapper.map_many(claims, session_id="promotion-demo")[0]
show(candidate.model_dump(mode="json"))

In [ ]:
default_result = engine.promote(candidate)
contradicted_result = engine.promote(candidate, context={"contradictions": ["ops-report-17"]})
ready_candidate = candidate.model_copy(update={"maturity": 3, "confidence": 0.92})
ready_result = engine.promote(ready_candidate)
strict_engine = PromotionEngine(
    rules=[
        MinimumConfidenceRule(0.9),
        MinimumMaturityRule(3),
        MinimumClaimsRule(1),
        NoContradictionsRule(),
    ]
)
strict_result = strict_engine.promote(ready_candidate)
show({
    "default_result": default_result.__dict__,
    "contradicted_result": contradicted_result.__dict__,
    "ready_result": ready_result.__dict__,
    "strict_result": strict_result.__dict__,
})

## What this notebook demonstrates

- candidate memories are not automatically trusted
- explicit governance rules gate promotion eligibility
- maturity and confidence can move a memory from blocked to promotable
- contradictions can keep otherwise-strong memories out of trusted memory